# HiProbCBM - Evaluasi ulang notebook 03-07

Notebook ini **tidak melatih** apa pun. Ia memuat `stage2_best.pth` yang sudah ada di Drive untuk skenario 03 (probcbm A2), 04 (probcbm A3), 05 (hicem utama), 06 (hicem A1), dan 07 (hicem A2), lalu menghitung ulang metrik test beserta intervensi (`intervention_delta` dan `intervention_curves.json`). Kode evaluasi identik dengan sel evaluasi notebook asal.

Jalankan semua sel dari atas ke bawah pada runtime GPU T4 (VPN dimatikan).


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import torch

DRIVE_ROOT = Path("/content/drive/MyDrive/TESIS AZHIM")
assert DRIVE_ROOT.is_dir(), f"Folder tesis tidak ditemukan: {DRIVE_ROOT}"
print("Drive siap:", DRIVE_ROOT)


## 2. Clone/perbarui repo dan instal dependensi (termasuk CLIP)

In [ ]:
%cd /content
!test -d HiProbCBM || git clone https://github.com/Azhimanich/HiProbCBM.git
%cd /content/HiProbCBM
!git pull --ff-only
!pip install -e ".[clip]" -q


## 3. Siapkan CUB dan PseudoKitchens dari Drive

In [ ]:
import os
import pickle

DRIVE_CUB_ROOT = DRIVE_ROOT / "CUB_200_2011"
assert (DRIVE_CUB_ROOT / "images").is_dir(), f"Folder CUB tidak ditemukan: {DRIVE_CUB_ROOT}"
metadata = DRIVE_CUB_ROOT / "metadata"
needed = [metadata / name for name in ("train.pkl", "val.pkl", "test.pkl")]
assert all(path.is_file() for path in needed), "metadata/train.pkl, val.pkl, test.pkl harus ada di Drive."

# Normalisasi ini idempotent: hanya menulis ulang pkl bila masih ada prefix lama.
marker = "CUB_200_2011/images/"
for path in needed:
    with path.open("rb") as handle:
        records = pickle.load(handle)
    changed = sum(marker in record["img_path"] for record in records)
    if changed:
        for record in records:
            if marker in record["img_path"]:
                record["img_path"] = record["img_path"].split(marker, 1)[1]
        with path.open("wb") as handle:
            pickle.dump(records, handle)
        print(f"Dinormalisasi: {path.name} ({changed} path)")
    else:
        print(f"OK: {path.name} ({len(records)} record)")

WORK_ROOT = Path("/content/HiProbCBM/datasets/CUB_200_2011")
WORK_ROOT.parent.mkdir(parents=True, exist_ok=True)
if WORK_ROOT.exists() or WORK_ROOT.is_symlink():
    assert WORK_ROOT.is_symlink() and WORK_ROOT.resolve() == DRIVE_CUB_ROOT.resolve(), (
        f"Symlink CUB tidak sesuai: {WORK_ROOT}. Jangan hapus folder non-symlink secara otomatis."
    )
else:
    os.symlink(DRIVE_CUB_ROOT, WORK_ROOT, target_is_directory=True)
print("CUB siap:", WORK_ROOT, "->", DRIVE_CUB_ROOT)


In [ ]:
import os

DRIVE_KITCHENS_ROOT = DRIVE_ROOT / "PseudoKitchens_processed"
assert (DRIVE_KITCHENS_ROOT / "manifest.csv").is_file(), (
    f"manifest.csv tidak ditemukan: {DRIVE_KITCHENS_ROOT}. Jalankan notebook setup/konversi PseudoKitchens dahulu."
)
assert (DRIVE_KITCHENS_ROOT / "info.json").is_file(), "info.json PseudoKitchens tidak ditemukan."

WORK_ROOT = Path("/content/HiProbCBM/datasets/PseudoKitchens")
WORK_ROOT.parent.mkdir(parents=True, exist_ok=True)
if WORK_ROOT.exists() or WORK_ROOT.is_symlink():
    assert WORK_ROOT.is_symlink() and WORK_ROOT.resolve() == DRIVE_KITCHENS_ROOT.resolve(), (
        f"Symlink PseudoKitchens tidak sesuai: {WORK_ROOT}. Jangan hapus folder non-symlink secara otomatis."
    )
else:
    os.symlink(DRIVE_KITCHENS_ROOT, WORK_ROOT, target_is_directory=True)
print("PseudoKitchens siap:", WORK_ROOT, "->", DRIVE_KITCHENS_ROOT)


## 4. Sambungkan train_log ke Drive dan cek GPU

In [ ]:
TRAIN_LOG_DRIVE = DRIVE_ROOT / "HiProbCBM_results" / "train_log"
TRAIN_LOG_DRIVE.mkdir(parents=True, exist_ok=True)
TRAIN_LOG_LOCAL = Path("/content/HiProbCBM/train_log")
if TRAIN_LOG_LOCAL.exists() or TRAIN_LOG_LOCAL.is_symlink():
    assert TRAIN_LOG_LOCAL.is_symlink() and TRAIN_LOG_LOCAL.resolve() == TRAIN_LOG_DRIVE.resolve(), (
        f"train_log lokal bukan symlink yang diharapkan: {TRAIN_LOG_LOCAL}"
    )
else:
    os.symlink(TRAIN_LOG_DRIVE, TRAIN_LOG_LOCAL, target_is_directory=True)

def assert_persistent_training_storage():
    assert Path("/content/drive/MyDrive").is_dir(), "Google Drive belum mounted."
    assert TRAIN_LOG_LOCAL.is_symlink(), "train_log harus symlink ke Drive."
    assert TRAIN_LOG_LOCAL.resolve() == TRAIN_LOG_DRIVE.resolve(), "Target train_log berubah."

assert_persistent_training_storage()
assert torch.cuda.is_available(), "GPU belum aktif. Pilih runtime GPU sebelum training."
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint Drive:", TRAIN_LOG_DRIVE)


## 5. Evaluasi ulang lima skenario

In [ ]:
import csv
import json
import statistics
from hiprobcbm.config import Config, load_config, prepare_log_dir, resolve_device
from hiprobcbm.engine.evaluate import evaluate_hiprobcbm_checkpoint, evaluate_hiprobcbm_interventions
from hiprobcbm.utils.seed import set_random_seed

# (notebook asal, PROTOKOL, VARIAN, config Tahap 1, config Tahap 2)
SKENARIO = [
    ("03", "probcbm", "a2", "configs/cub_stage1_probcbm_reported.yaml", "configs/cub_stage2_probcbm_reported.yaml"),
    ("04", "probcbm", "a3", "configs/cub_stage1_probcbm_reported_batchtopk.yaml", "configs/cub_stage2_probcbm_reported_batchtopk.yaml"),
    ("05", "hicem", "utama", "configs/kitchens_stage1_hicem_reported.yaml", "configs/kitchens_stage2_hicem_reported.yaml"),
    ("06", "hicem", "a1", "configs/kitchens_stage1_hicem_reported.yaml", "configs/kitchens_stage2_hicem_reported.yaml"),
    ("07", "hicem", "a2", "configs/kitchens_stage1_hicem_reported.yaml", "configs/kitchens_stage2_hicem_reported.yaml"),
]
SEEDS = [42, 43, 44]

def stage1_dir(seed):
    induk = "a3" if VARIAN == "a3" else "utama"  # A1/A2 memakai hierarchy utama
    return prepare_log_dir(load_config(STAGE1_CONFIG), f"train_log/{PROTOKOL}/{induk}/seed_{seed}")

def stage2_dir(seed):
    values = load_config(STAGE2_CONFIG).to_dict()
    if VARIAN in {"a1", "a2"}:
        values["experiment_name"] = f"{values['experiment_name']}_ablation_{VARIAN}"
    return prepare_log_dir(Config(values), f"train_log/{PROTOKOL}/{VARIAN}/seed_{seed}")

for NOTEBOOK, PROTOKOL, VARIAN, STAGE1_CONFIG, STAGE2_CONFIG in SKENARIO:
    print(f"\n{'=' * 72}\nNotebook {NOTEBOOK}: {PROTOKOL} / {VARIAN}\n{'=' * 72}")
    assert_persistent_training_storage()
    device = resolve_device(0)
    rows = []
    for seed in SEEDS:
        parent_dir, child_dir = stage1_dir(seed), stage2_dir(seed)
        best = child_dir / "stage2_best.pth"
        manifest = child_dir / "run_manifest.json"
        hierarchy_path = parent_dir / "pseudo_hierarchy.pt"
        if not (best.exists() and manifest.exists() and hierarchy_path.exists()):
            print(f"LEWATI seed {seed}: checkpoint atau hierarchy belum lengkap.")
            continue
        cfg = Config(json.loads(manifest.read_text(encoding="utf-8"))["config"])
        set_random_seed(cfg.get("seed", seed))
        hierarchy = torch.load(hierarchy_path, map_location="cpu", weights_only=True)
        metrics = evaluate_hiprobcbm_checkpoint(cfg, best, hierarchy["subconcepts_per_concept"], device)
        interventions = evaluate_hiprobcbm_interventions(cfg, best, hierarchy["subconcepts_per_concept"], device)
        metrics["intervention_delta"] = interventions["intervention_delta"]
        (child_dir / "intervention_curves.json").write_text(json.dumps(interventions, indent=2), encoding="utf-8")
        rows.append({"protokol": PROTOKOL, "varian": VARIAN, "seed": seed, "checkpoint": str(best), **metrics})
        print(f"OK seed {seed}: {metrics}")

    out_csv = Path("/content/HiProbCBM") / "train_log" / PROTOKOL / VARIAN / "ringkasan_tiga_seed.csv"
    if rows:
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        fields = list(dict.fromkeys(key for row in rows for key in row))
        with out_csv.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=fields)
            writer.writeheader()
            writer.writerows(rows)
        print("CSV:", out_csv)
        for metric in sorted({key for row in rows for key, value in row.items() if isinstance(value, (int, float)) and key != "seed"}):
            values = [row[metric] for row in rows if isinstance(row.get(metric), (int, float))]
            mean = statistics.mean(values)
            std = statistics.stdev(values) if len(values) > 1 else 0.0
            print(f"{metric}: {mean:.6f} +/- {std:.6f} (n={len(values)})")
    else:
        print("Belum ada tiga checkpoint yang siap dievaluasi.")
